# Customer Churn Prediction using Random Tree & Random Forest Classifiers

**Objective**: Build a machine learning classification pipeline to predict customer churn in subscription-based services using Decision Trees and Random Forest ensembles.

---

## 1. Import Libraries & Load Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

# Load generated churn dataset
df = pd.read_csv('../data/customer_churn.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Churn Rate Distribution
print("Target Class Distribution (0 = Retained, 1 = Churned):")
print(df['churn'].value_counts(normalize=True))

# Numerical feature statistics
df.describe()

## 3. Data Preprocessing & Encoding

In [ ]:
categorical_cols = [
    'gender', 'partner', 'dependents', 'contract_type',
    'internet_service', 'tech_support', 'online_security',
    'online_backup', 'device_protection', 'streaming_tv',
    'streaming_movies', 'paperless_billing', 'payment_method'
]

encoded_df = df.copy()
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    encoded_df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

X = encoded_df.drop(columns=['customer_id', 'churn'])
y = encoded_df['churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]} | Testing set size: {X_test.shape[0]}")

## 4. Train Random Tree & Random Forest Classifiers

In [ ]:
# 1. Train Decision Tree (Single Random Tree)
dt_classifier = DecisionTreeClassifier(max_depth=5, criterion='gini', random_state=42)
dt_classifier.fit(X_train, y_train)
dt_preds = dt_classifier.predict(X_test)

# 2. Train Random Forest Classifier
rf_classifier = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf_classifier.fit(X_train, y_train)
rf_preds = rf_classifier.predict(X_test)
rf_probas = rf_classifier.predict_proba(X_test)[:, 1]

print("=== Random Forest Classification Report ===")
print(classification_report(y_test, rf_preds))

## 5. Feature Importance Analysis

In [ ]:
importances = pd.Series(rf_classifier.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
importances.plot(kind='bar', color='#06B6D4')
plt.title('Random Forest Feature Importance for Customer Churn')
plt.ylabel('Relative Gini Importance Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()